<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 28px; border-radius: 10px; color: #0f172a; font-family: sans-serif;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Pipeline Stage 05
    </span>
    <h1 style="color: #0f172a; margin-top: 10px; margin-bottom: 8px; font-size: 26px; border-bottom: none;">
        Exploratory Data Analysis & Feature Validation
    </h1>
    <p style="color: #475569; font-size: 14px; margin-bottom: 20px;">
        Executes the EDA and data validation phase for the multi-asset feature matrix, focusing on structural sanity checks, logical bounding, and removing mathematical redundancies for tree-based models.
    </p>
    <div style="background-color: #f1f5f9; padding: 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: bold; text-transform: uppercase;">Key Objectives in this Module:</p>
        <ol style="margin-top: 8px; margin-bottom: 0; padding-left: 20px; color: #334155; font-size: 13px; line-height: 1.6;">
            <li><b>Dataset Verification:</b> Inspect fundamental structure, shape, and data types across the stacked matrix.</li>
            <li><b>Redundancy Elimination:</b> Identify and remove zero-variance features and highly correlated pairs (&rho; &gt; 0.90).</li>
            <li><b>Distribution Sanity Checks:</b> Verify outlier boundaries and extreme events without forcing statistical normality.</li>
            <li><b>Visual Unit Testing:</b> Validate time-series logic (like roll calculations and drawdowns) across random asset samples.</li>
            <li><b>Pooling Validity Test:</b> Confirm whether each feature's distribution genuinely differs by asset class (Kruskal-Wallis) or the single pooled model holds.</li>
            
<li><b>Small-Class Ticker Check:</b> Confirm findings for thin asset classes (2-4 tickers) reflect every member, not one atypical instrument.</li>
        </ol>
    </div>
    <p style="margin-top: 15px; margin-bottom: 0; color: #64748b; font-size: 12px;">
        <b>Next Milestones:</b> Target Engineering (t+20) &rarr; Walk-Forward Splitting &rarr; Model Training
    </p>
</div>

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os
from pathlib import Path
sys.path.append(os.path.abspath(".."))

import pandas as pd
from src.config import MASTER_FILE, FEATURE_DIR
from src.eda_qa import (get_dataset_overview, get_missing_values_by_class, check_logical_bounds, find_redundant_features, compare_feature_distributions_by_class, per_ticker_summary_for_small_classes)
from src.eda_plots import (plot_feature_distributions, plot_time_series_sample, plot_correlation_heatmap, plot_distributions_by_class)

In [ ]:
# Load the engineered dataset
df = pd.read_parquet(MASTER_FILE)
print(f"Data loaded successfully. Shape: {df.shape}")

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 01 & 02
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Dataset Overview & Summary Statistics
    </h3>
    <p style="color: #475569; font-4e: 13.5px; margin-bot.5tom: 14px; line-height: 1.5;">
        Inspects the fundamental structure, shape, and high-level statistical summary of the concatenated feature matrix.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; f3-size: 12.5px; line-height: 1.6;">
            <li>Ensures all ticker data was concatenated successfully without index misalignment.</li>
            <li>Identifies glaring calculation errors, infinite values (<code>np.inf</code>), or impossible boundaries (e.g., negative standard deviations) prior to deeper visualizations.</li>
        </ul>
    </div>
</div>

In [ ]:
# 1. Dataset Overview
overview = get_dataset_overview(df)
print("=== Dataset Overview ===")
for key, value in overview.items():
    print(f"{key}: {value}")

print("\n=== Logical Boundary Checks ===")
display(check_logical_bounds(df))

In [ ]:
# 2. Description table
df.describe(include='all')

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 03
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Missing Value Analysis
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        Evaluates the dataset for missing values (<code>NaN</code>). In financial time-series, missing values generally fall into two categories:
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>Lookback NaNs:</b> Expected missing rows at the beginning of an asset's history due to rolling calculations (e.g., a 60-day moving average requires 59 days of history).</li>
            <li><b>Structural NaNs:</b> Missing data inherent to an asset class (e.g., Bonds and Real Estate lacking daily volume metrics). Tree-based models handle these structural gaps natively.</li>
        </ul>
    </div>
</div>

In [ ]:
# 3. Missing Value Analysis
print("=== % Missing Values by Asset Class ===")
missing_report = get_missing_values_by_class(df)
missing_count = df.isna().sum()
display(missing_report)
print(missing_count)

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 04
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Global Feature Correlation & Zero Variance
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        Evaluates feature redundancy using the Spearman Rank correlation matrix and scans for zero-variance features (constants).
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li>Unlike Pearson (which assumes linear relationships), Spearman evaluates the <b>monotonic relationship</b> between variables.</li>
            <li>This approach perfectly mirrors how tree-based algorithms split data based on rank-ordering rather than linear distance.</li>
        </ul>
    </div>
</div>

In [ ]:
# 4. Correlation & Zero Variance Checks
zero_var, high_corr = find_redundant_features(df, corr_threshold=0.90)

print(f"Zero Variance Features (Consider Removing): {zero_var}")
print("\n=== Highly Correlated Pairs (|ρ| > 0.90) ===")
display(high_corr)

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Feature Reduction Rationale
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        To prevent multicollinearity, eliminate redundant signals, and streamline the feature matrix for tree-based models (Candidate models: LightGBM and XGBoost). While extreme correlations dilute feature importance and slow down training, we intentionally enforce a strict drop threshold (&rho; &gt; 0.90) to eliminate only mathematically identical features.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>Dropped (Redundant):</b> <code>Open, High, Low</code> (redundant to Close), <code>SMA20_Ratio, SMA60_Ratio</code> (redundant to EMAs), and <code>MACD, MACD_Signal</code> (encapsulated by MACD_Histogram) &mdash; the exact pairs and &rho; values are in the table above.</li>
        </ul>
    </div>
</div>

In [ ]:
# Drop redundant raw price columns, overlapping moving averages, and MACD components
cols_to_drop = [
    "Open",
    "High",
    "Low",
    "SMA20_Ratio",
    "SMA60_Ratio",
    "MACD",
    "MACD_Signal",
]

cols_to_drop = [col for col in cols_to_drop if col in df.columns]

df = df.drop(columns=cols_to_drop)
print(f"Successfully dropped {len(cols_to_drop)} redundant features: {cols_to_drop}")

In [ ]:
#!pip install missingno

In [ ]:
import missingno as msno
msno.matrix(df)

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 05
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Distribution & Outlier Analysis
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        Visualizes the distributions (histograms) and outliers (boxplots) for engineered features to verify market realities and data integrity.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>Objective:</b> Focus on structural sanity rather than standard bell-curve normality.</li>
            <li>Tree-based algorithms (Candidate models: LightGBM and XGBoost) are unaffected by skewness or extreme values, requiring no transformations.</li>
        </ul>
    </div>
</div>

In [ ]:
# 5. Distributions and Outliers
# Select a mix of price features, momentum, and technical indicators to check for skew/outliers
features_to_check = [
    # Momentum & Trend
    "Log_Return_1D",
    "Log_Return_5D",
    "Log_Return_20D",
    "Log_Return_60D",
    "RSI14",
    "MACD_Histogram",
    "EMA20_Ratio",
    "EMA50_Ratio",
    
    # Volatility & Risk
    "Volatility_20D",
    "Volatility_60D",
    "Bollinger_Width",
    "Max_Drawdown_60D",
    
    # Volume Dynamics
    "Volume_Change",
    "Relative_Volume",
    "OBV_Z_Score"
]
print("=== Feature Distributions & Boxplots ===")
stocks_df = df[df["Asset_Class"].str.lower() == "crypto"].copy()
plot_feature_distributions(df, features_to_check)

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
<span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
Key Takeaway
</span>
<h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
Feature Distribution & Outlier Analysis
</h3>
<p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
The histograms (distributions) and boxplots (outliers) above serve as a visual sanity check for our engineered features. Rather than looking for perfect bell curves, we are verifying structural bounds, market realities, and data integrity prior to modeling.
</p>
<div style="background-color: #f1f5f9; padding: 16px; border-radius: 8px; border-left: 4px solid #2563eb; color: #334155; font-size: 13px; line-height: 1.6;">
<b style="color: #0f172a; font-size: 14px;">1. Momentum & Trend Features</b><br>
&bull; <b>Features:</b> <code>Log_Return_1D</code>, <code>5D</code>, <code>20D</code>, <code>60D</code>, <code>RSI14</code>, <code>MACD_Histogram</code>, <code>EMA20_Ratio</code>, <code>EMA50_Ratio</code><br>
&bull; <b>What they measure:</b> The speed, magnitude, and direction of price changes. They capture short-term mean-reversion and long-term trend strength.<br>
&bull; <b>What to look for:</b> Returns and MACD should be roughly symmetric and centered around 0. RSI must be strictly bounded between 0 and 100.<br>
&bull; <b>Takeaway:</b> Financial returns display "fat tails" (excess kurtosis). The massive outlier dots on the boxplots represent real market shocks (e.g., flash crashes, earnings gaps). These are valid signals, not errors.<br><br>
<b style="color: #0f172a; font-size: 14px;">2. Volatility & Risk Features</b><br>
&bull; <b>Features:</b> <code>Volatility_20D</code>, <code>Volatility_60D</code>, <code>Bollinger_Width</code>, <code>Max_Drawdown_60D</code><br>
&bull; <b>What they measure:</b> Market turbulence, price dispersion, and peak-to-trough historical losses.<br>
&bull; <b>What to look for:</b> Volatility and Bollinger Width must be strictly positive (&gt; 0) and will naturally be heavily right-skewed. Max Drawdown must be strictly non-positive (&lt;= 0) and will be left-skewed.<br>
&bull; <b>Takeaway:</b> The long tails in these charts represent crisis periods and high-risk regimes. Because our target model will predict future volatility, these extreme historical risk events are crucial for the algorithm to learn from.<br><br>
<b style="color: #0f172a; font-size: 14px;">3. Volume Dynamics</b><br>
&bull; <b>Features:</b> <code>Volume_Change</code>, <code>Relative_Volume</code>, <code>OBV_Z_Score</code><br>
&bull; <b>What they measure:</b> Liquidity, institutional participation, and buying/selling pressure relative to historical baselines.<br>
&bull; <b>What to look for:</b> Relative volume should cluster tightly around 1.0. OBV Z-Score should be roughly centered around 0.<br>
&bull; <b>Takeaway:</b> Volume data is prone to explosive spikes (e.g., 500% surges on news). Keep in mind that certain asset classes (Bonds, Real Estate) naturally lack daily volume data, which will be handled gracefully by our target algorithms.<br>
<hr style="border: 0; height: 1px; background-color: #cbd5e1; margin: 16px 0;">
<b style="color: #0f172a; font-size: 14px;">Why We Skip Smoothing, Scaling, and Log-Transformations</b><br>
In classical statistics or linear regression, heavily skewed data and massive outliers require log-transformations, winsorization (clipping), or Gaussian smoothing to force a normal distribution. <b>We are intentionally skipping these steps for the following reasons:</b><br><br>
1. <b>Tree Models are Invariant to Monotonic Transformations:</b> Our target candidate algorithms (LightGBM and XGBoost) split data based on rank-ordering rather than absolute linear distance. A log transformation does not change the rank order of the data, meaning it would yield the exact same tree structure.<br>
2. <b>Preserving Risk Signals:</b> If we apply smoothing techniques or clip the outliers, we destroy the exact high-volatility spikes (market crashes) the model needs to see to accurately predict our Risk Target (20-Day Realized Volatility).<br>
3. <b>Inherent Smoothing:</b> Features like our 60-day moving averages and 20-day volatilities are <i>already</i> aggregated and smoothed mathematical rolling windows. Applying further smoothing would cause the model to react too slowly to sudden market shifts.
</div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 06
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Time-Series Behavior & Visual Unit Testing
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        Executes a visual unit test by plotting a random sample of assets from different classes on a shared timeline.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li>Visually verifies that mathematical rolling window calculations maintain structural integrity across historically distinct market environments.</li>
        </ul>
    </div>
</div>

In [ ]:
# 6. Time-Series Behavior over Time
# Randomly samples 1 asset from Stocks, ETFs, Crypto, Bonds, Real Estate, and Commodities
# Plots them together to ensure feature logic (like Max Drawdown) makes visual sense

ts_features = [
    "Close", 
    "Volatility_20D",
    "RSI14", 
    "Max_Drawdown_60D"
]

print("=== Time-Series Validation ===")
plot_time_series_sample(df, features=ts_features, n_assets_per_class=1)

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Time-Series Validation
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        This random sampling confirms that our feature engineering logic behaves correctly across entirely different asset environments (e.g., Crypto vs. Bonds).
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li style="margin-bottom: 6px;"><b><code>Close</code> (Price Trajectory):</b> Verifies that our data provider's adjusted prices are continuous. We are looking for an absence of sudden, illogical gaps or flatlines that would indicate stock splits or corporate actions were not properly handled.</li>
            <li style="margin-bottom: 6px;"><b><code>Volatility_20D</code> (Risk Regimes):</b> Confirms that our rolling standard deviation accurately captures known historical stress periods. We should see distinct spikes during market panics and flat, low baselines during stable bull markets.</li>
            <li style="margin-bottom: 6px;"><b><code>RSI14</code> (Momentum Bounds):</b> Verifies our bounding math. The indicator properly oscillates back and forth based on price velocity and never breaches the 0 to 100 structural limits.</li>
            <li><b><code>Max_Drawdown_60D</code> (Downside Exposure):</b> Validates our peak-to-trough logic. The values correctly anchor at <code>0.0</code> during new highs and only extend downward into negative territory during sell-offs, confirming the directional sign is mathematically correct.</li>
        </ul>
    </div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 07
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Global Feature Correlation (Spearman Rank)
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 0px; line-height: 1.5;">
        To evaluate feature redundancy, we use the Spearman Rank correlation. Unlike Pearson (which assumes linear relationships), Spearman evaluates the monotonic relationship between variables. This is specifically chosen because tree-based models split data based on rank-ordering rather than linear distance.
    </p>
</div>

In [ ]:
# 7. Global Feature Correlation
# Identifies highly redundant features across the entire stacked dataset using Spearman rank

print("=== Feature Correlation Heatmap ===")
plot_correlation_heatmap(df, method="spearman")

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Reading the Full Heatmap
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        The heatmap's real value is everything Step 04 couldn't show &mdash; the 0.80&ndash;0.89 range, invisible to a &gt; 0.90 filter but visible here.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>Preserving Nuance:</b> Features in the 0.80 - 0.89 range (such as short-term vs. medium-term volatility, or RSI vs. EMA Ratios) are kept, not dropped. While they track the same underlying market factors (like momentum or risk), their different mathematical constructions (bounded vs. unbounded, reactive vs. smoothed) provide unique, non-redundant split opportunities for our decision trees.</li>
        </ul>
    </div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 08
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Feature Distributions by Asset Class
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        Every check so far treats the full universe as one pool. This step checks whether that pooling assumption actually holds: does each feature mean roughly the same thing across Equity, Crypto, Bond, Real Estate, Commodity, and ETF, or does one class sit on a completely different part of the distribution?
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li style="margin-bottom: 6px;"><b>Why it matters:</b> The downstream model is a single global model trained on all asset classes together, with <code>Asset_Class</code> as a categorical feature. If a feature exhibits substantially different distributions across asset classes, this indicates cross-sectional heterogeneity that the global model may need to account for through asset-class effects and feature interactions.</li>
            <li style="margin-bottom: 6px;"><b>Hypothesis:</b> Tested separately per feature. H₀ &mdash; all six asset classes share the same underlying distribution, any apparent gap is sampling noise. H₁ &mdash; at least one class genuinely differs.</li>
            <li style="margin-bottom: 6px;"><b>Why Kruskal-Wallis:</b> Chosen over a parametric ANOVA because it doesn't assume normality &mdash; and return/risk data is well known to be fat-tailed rather than normal.</li>
            <li><b>Dependence mitigation:</b> To reduce pseudo-replication caused by serial dependence, the test uses one aggregated statistic per ticker rather than treating daily observations as independent. i.e The test runs on one median per ticker, not every daily row, since daily rows within a ticker are heavily autocorrelated for engineered features (a 20-day rolling feature shares 19 of 20 underlying days with the row next to it) and would otherwise inflate significance. Bond and RealEstate are excluded from the significance test only &mdash; one instrument can't represent a distribution.</li>
        </ul>
    </div>
</div>    </div>
</div>

In [ ]:
divergence_report = compare_feature_distributions_by_class(df, features_to_check)
print("=== Feature Divergence Across Asset Classes ===")
display(divergence_report)

plot_distributions_by_class(df, features_to_check)

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        How to Read This
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        Interpreting the Kruskal-Wallis table and the boxplots to determine if an asset class sits in a completely different feature regime than the rest of the universe.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 13px; line-height: 1.6;">
            <li style="margin-bottom: 6px;"><b>The Table:</b> <code>Median_Spread_Normalized</code> is used as a descriptive measure of the magnitude of cross-class differences, while the Kruskal-Wallis p-value provides complementary evidence of statistical separation. The resulting p-values are more interpretable than those obtained from treating all daily observations as independent, although they remain exploratory because the number of independent tickers is small and cross-ticker dependence may remain. With the per-ticker fix in place, <code>P_Value</code> ranges 0.001 to 0.27, and agrees with <code>Median_Spread_Normalized</code>: every feature above the ~1.0 spread threshold is significant (p &lt; 0.05), every feature below it isn't. Still lead with <code>Median_Spread_Normalized</code> regardless &mdash; it's descriptive, so it never depended on the assumption that just got fixed. Above ~1-2 means a genuinely different regime; near 0 means the classes are practically interchangeable.</li>
            <li style="margin-bottom: 6px;"><b>The Boxplots (Outliers):</b> Look for a class sitting off on its own, box barely overlapping the rest &mdash; Crypto on volatility/return features is the obvious candidate, given how much more volatile it typically is than Bonds or Real Estate.</li>
            <li style="margin-bottom: 6px;"><b>The Boxplots (Squashed):</b> Look for a class with a near-flat, squashed box. This is more concerning than a wide one, since it can mean a feature is close to constant for that class. Worth checking the synthetic assets specifically for this.</li>
            <li style="margin-bottom: 6px;"><b>What to do with it:</b> This isn't a pooling-vs-not-pooling decision &mdash; going back to six separate models undoes the reason why pooling was choosen in the first place. If a handful of features show a large spread for one class, simply confirm those features still carry useful signal <i>within</i> that class during model evaluation (per-class IC check), rather than restructuring the modeling approach now.</li>
            <li><b>Verdict for this run:</b> Six features clear both bars (p &lt; 0.05 and spread &gt; 1.0): <code>RSI14</code>, <code>Volatility_20D</code>, <code>Volatility_60D</code>, <code>Bollinger_Width</code>, <code>Max_Drawdown_60D</code>, and <code>Relative_Volume</code>. Five of these are exactly the risk features &mdash; Crypto being riskiest and Bond safest is the expected result, not a problem. <code>Relative_Volume</code> is the smaller, separate case (ETF trading more actively than Commodity), worth a light spot-check but not urgent. <code>RSI14</code>'s Bond value of exactly 100 is structural (no down days to average against), not a genuine momentum reading &mdash; worth remembering if this number gets quoted later. Every other feature &mdash; all four return horizons, both EMA ratios, <code>OBV_Z_Score</code>, <code>Volume_Change</code>, <code>MACD_Histogram</code> &mdash; shows no significant difference and low spread: the pooling assumption holds cleanly for them. Nothing here needs to be dropped or transformed. This step is closed, with one item carried forward: confirm the risk features and <code>Relative_Volume</code> still carry predictive signal within Crypto, Bond, and Commodity specifically, once per-class model evaluation happens.</li>
        </ul>
    </div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Sanity Check
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Small-Class Ticker Check
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        Cross-ticker variation within a class is the actual signal the model learns from &mdash; the model never sees an individual ticker identity, only <code>Asset_Class</code>, precisely because pooling was chosen over training a separate model per instrument. Checking whether each stock "matches" the Stocks average would be checking for the opposite of what a predictive model needs.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li style="margin-bottom: 6px;"><b>What this checks:</b> whether a Step 08 finding for a small class is really true of every member, or is being driven by just one. Scoped to classes with 5 or fewer tickers &mdash; For classes with more tickers it doesn't have this risk and is excluded automatically.</li>
            <li><b>How to read it:</b> for 2 &lt; <code>Asset_Class</code> &gt; 5, one median sitting clearly apart from the other three is worth a second look. For <code>Asset_Class</code> with 2 tickers, there's no statistical way to call either one the "outlier" &mdash; just sanity-check the gap against what the two instruments actually are.</li>
        </ul>
    </div>
</div>

In [ ]:
per_ticker_summary_for_small_classes(df, features_to_check)

In [ ]:
output_file = FEATURE_DIR / "clean_features_after_EDA.parquet"
df.to_parquet(output_file, index=False)
output_file_csv = FEATURE_DIR / "clean_features_after_EDA.csv"
df.to_csv(output_file, index=False)